# Two-Phase YOLO26n Pipeline Runner

This notebook runs the Sprint 4 two-phase pipeline in English, using a crop-stage YOLO weapon detector checkpoint for Stage 2 and a full-frame checkpoint for the baseline comparison.

- Stage 2 weapon detection inside every detected person crop
- the optional `hold` / `no_hold` gate only when explicitly enabled
- the single-stage baseline comparison on the full test split

Workflow:

1. Build Phase 1 person crops
2. Optional: train the `hold` / `no_hold` classifier
3. Run two-phase inference on a split
4. Compare two-phase vs single-stage YOLO26n


In [8]:
from pathlib import Path
import json
import os
import subprocess
import sys
import textwrap

import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
assert (PROJECT_ROOT / 'scripts').exists(), 'Run this notebook from the repository root.'

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

def run_command(args):
    """Run a project command and print stdout/stderr before failing."""
    args = [str(arg) for arg in args]
    print('Running:')
    print(' '.join(args))
    print()

    result = subprocess.run(
        args,
        cwd=PROJECT_ROOT,
        text=True,
        capture_output=True
    )

    print('STDOUT:')
    print(result.stdout)

    print('STDERR:')
    print(result.stderr)

    print('RETURN CODE:', result.returncode)
    result.check_returncode()


def require_cuda_device(gpu_index=0):
    """Return an Ultralytics-compatible CUDA device string, or fail with a clear diagnosis."""
    import torch

    print('Python executable:', sys.executable)
    print('Torch version:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    print('CUDA device count:', torch.cuda.device_count())
    print("CUDA_VISIBLE_DEVICES:", os.environ.get('CUDA_VISIBLE_DEVICES'))

    if not torch.cuda.is_available() or torch.cuda.device_count() <= gpu_index:
        raise RuntimeError(textwrap.dedent(f"""
        GPU was requested, but this notebook kernel is not using a CUDA-enabled PyTorch build.

        Current Python:
            {sys.executable}

        Current torch:
            {torch.__version__}

        What to fix:
        1. Install the CUDA-enabled PyTorch build in this exact environment, or
        2. Change the Jupyter kernel to the environment where CUDA already works.

        Validation after fixing:
            import torch
            print(torch.__version__)
            print(torch.cuda.is_available())
            print(torch.cuda.get_device_name(0))

        Expected result:
            torch.cuda.is_available() == True
        """).strip())

    print('Selected GPU:', torch.cuda.get_device_name(gpu_index))
    # Ultralytics expects GPU ids as "0", "1", etc.
    return str(gpu_index)


def show_csv(path, rows=10):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Missing file: {path}')
    df = pd.read_csv(path)
    print(f'{path} -> {len(df)} rows')
    return df.head(rows)


def show_text_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Missing file: {path}')
    display(Markdown(path.read_text(encoding='utf-8')))


In [9]:
# Main parameters
FORCE_GPU = True
GPU_INDEX = 0
DEVICE = require_cuda_device(GPU_INDEX) if FORCE_GPU else 'cpu'

SPLIT = 'test'
OUTPUT_PREFIX = SPLIT
OVERWRITE_CROPS = True

# Optional overrides
PERSON_MODEL = None
ENABLE_HOLD_GATE = False
HOLD_CHECKPOINT = PROJECT_ROOT / 'runs' / 'two_phase' / 'carry_classifier' / 'best.pt'

# Use your crop-stage YOLO best.pt here.
DEFAULT_WEAPON_MODEL = PROJECT_ROOT / 'runs' / 'two_phase' / 'weapon_crop_detector' / 'weights' / 'best.pt'
WEAPON_MODEL = DEFAULT_WEAPON_MODEL if DEFAULT_WEAPON_MODEL.exists() else Path('runs/single_stage/yolo26n_img9604/weights/best.pt')

# Optional training overrides. Keep as None to use configs/two_phase.yaml defaults./nEPOCHS = None
BATCH_SIZE = None
NUM_WORKERS = None

TWO_PHASE_PREDICTIONS = PROJECT_ROOT / 'runs' / 'two_phase' / 'predictions' / f'{OUTPUT_PREFIX}_predictions.csv'
TWO_PHASE_IMAGE_SUMMARY = PROJECT_ROOT / 'runs' / 'two_phase' / 'predictions' / f'{OUTPUT_PREFIX}_image_summary.csv'
TWO_PHASE_PIPELINE_SUMMARY = PROJECT_ROOT / 'runs' / 'two_phase' / 'predictions' / f'{OUTPUT_PREFIX}_pipeline_summary.csv'
EVAL_COMPARISON = PROJECT_ROOT / 'runs' / 'two_phase' / 'evaluation' / f'{OUTPUT_PREFIX}_comparison.csv'
EVAL_SUMMARY = PROJECT_ROOT / 'runs' / 'two_phase' / 'evaluation' / f'{OUTPUT_PREFIX}_summary.md'

print('Project root:', PROJECT_ROOT)
print('Split:', SPLIT)
print('Device:', DEVICE)
print('Weapon model:', WEAPON_MODEL)


Python executable: .venv\Scripts\python.exe
Torch version: 2.7.1+cu118
CUDA available: True
CUDA device count: 1
CUDA_VISIBLE_DEVICES: None
Selected GPU: NVIDIA GeForce RTX 3060 Laptop GPU
Project root: 
Split: test
Device: 0
Weapon model: runs/single_stage\yolo26n_img9604\weights\best.pt


In [10]:
required_paths = [
    PROJECT_ROOT / 'scripts' / 'build_two_phase_dataset.py',
    PROJECT_ROOT / 'scripts' / 'train_carry_classifier.py',
    PROJECT_ROOT / 'scripts' / 'run_two_phase_inference.py',
    PROJECT_ROOT / 'scripts' / 'evaluate_detection_pipeline.py',
    PROJECT_ROOT / 'configs' / 'two_phase.yaml',
    PROJECT_ROOT / 'data' / 'splits' / f'{SPLIT}_manifest.csv',
]

missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required files:\n- ' + '\n- '.join(missing))

if not Path(WEAPON_MODEL).exists():
    raise FileNotFoundError(
        'WEAPON_MODEL does not exist. Update the parameter cell with the path to your YOLO26n best.pt checkpoint.'
    )

print('All required files were found.')


All required files were found.


## Step 1. Build the Phase 1 person crops

This generates both the optional `hold` / `no_hold` crops and the `yolo_crops/` export used to train the crop-stage weapon detector.


In [11]:
cmd = [sys.executable, 'scripts/build_two_phase_dataset.py', '--device', DEVICE]

if OVERWRITE_CROPS:
    cmd.append('--overwrite')

if PERSON_MODEL is not None:
    cmd.extend(['--person-model', PERSON_MODEL])

run_command(cmd)


Running:
.venv\Scripts\python.exe scripts/build_two_phase_dataset.py --device 0 --overwrite/n
STDOUT:
[OK] train: images=3683 hold_crops=885 no_hold_crops=3025 stage0_miss_images=48
[OK] val: images=435 hold_crops=89 no_hold_crops=407 stage0_miss_images=5
[OK] test: images=1031 hold_crops=1466 no_hold_crops=1093 stage0_miss_images=42
[OK] Saved: data/interim\two_phase\metadata/two_phase_dataset_summary.csv/n[OK] Saved: docs/sprint4_two_phase_dataset_summary.md/n
STDERR:

RETURN CODE: 0


In [12]:
metadata_root = PROJECT_ROOT / 'data' / 'interim' / 'two_phase' / 'metadata'
for split_name in ['train', 'val', 'test']:
    crop_csv = metadata_root / f'{split_name}_person_crops.csv'
    miss_csv = metadata_root / f'{split_name}_stage0_misses.csv'
    print('\n===', split_name.upper(), '===')
    if crop_csv.exists():
        crop_df = pd.read_csv(crop_csv)
        print('crops:', len(crop_df), '| labels:', crop_df['label'].value_counts().to_dict() if not crop_df.empty else {})
    else:
        print('Missing:', crop_csv)
    if miss_csv.exists():
        miss_df = pd.read_csv(miss_csv)
        print('stage0 misses:', len(miss_df))
    else:
        print('Missing:', miss_csv)



=== TRAIN ===
crops: 3910 | labels: {'no_hold': 3025, 'hold': 885}
stage0 misses: 48

=== VAL ===
crops: 496 | labels: {'no_hold': 407, 'hold': 89}
stage0 misses: 5

=== TEST ===
crops: 2559 | labels: {'hold': 1466, 'no_hold': 1093}
stage0 misses: 42


## Step 2. Optional: train the `hold` / `no_hold` classifier

This writes the best checkpoint to `runs/two_phase/carry_classifier/best.pt`.


In [13]:
cmd = [sys.executable, 'scripts/train_carry_classifier.py', '--device', DEVICE]

if EPOCHS is not None:
    cmd.extend(['--epochs', str(EPOCHS)])
if BATCH_SIZE is not None:
    cmd.extend(['--batch-size', str(BATCH_SIZE)])
if NUM_WORKERS is not None:
    cmd.extend(['--num-workers', str(NUM_WORKERS)])

run_command(cmd)


Running:
.venv\Scripts\python.exe scripts/train_carry_classifier.py --device 0/n
STDOUT:
[Epoch 1/12] train_loss=0.9502 val_loss=0.7101 best_f1_thr=0.34 best_f1=0.5015 gate_thr=0.34 gate_recall=0.9551 gate_f1=0.5015
[Epoch 2/12] train_loss=0.7865 val_loss=0.7204 best_f1_thr=0.71 best_f1=0.5202 gate_thr=0.62 gate_recall=0.8202 gate_f1=0.5177
[Epoch 3/12] train_loss=0.7149 val_loss=0.8164 best_f1_thr=0.83 best_f1=0.5050 gate_thr=0.72 gate_recall=0.7303 gate_f1=0.5039
[Epoch 4/12] train_loss=0.6760 val_loss=0.6466 best_f1_thr=0.66 best_f1=0.5753 gate_thr=0.66 gate_recall=0.7079 gate_f1=0.5753
[Epoch 5/12] train_loss=0.6458 val_loss=0.6375 best_f1_thr=0.66 best_f1=0.5584 gate_thr=0.40 gate_recall=0.8764 gate_f1=0.5552
[Epoch 6/12] train_loss=0.6135 val_loss=0.6406 best_f1_thr=0.69 best_f1=0.5764 gate_thr=0.69 gate_recall=0.7416 gate_f1=0.5764
[Epoch 7/12] train_loss=0.5723 val_loss=0.7205 best_f1_thr=0.75 best_f1=0.5804 gate_thr=0.75 gate_recall=0.7303 gate_f1=0.5804
[Epoch 8/12] train_los

In [14]:
carry_dir = PROJECT_ROOT / 'runs' / 'two_phase' / 'carry_classifier'
metrics_path = carry_dir / 'metrics.csv'
summary_path = carry_dir / 'summary.md'
checkpoint_path = carry_dir / 'best.pt'

print('Checkpoint exists:', checkpoint_path.exists(), checkpoint_path)
display(show_csv(metrics_path, rows=12))


Checkpoint exists: True runs/two_phase\carry_classifier\best.pt
runs/two_phase\carry_classifier\metrics.csv -> 104 rows


,kind,epoch,split,loss,train_loss,best_f1_threshold,best_f1_precision,best_f1_recall,best_f1_f1,stage1_gate_threshold,stage1_gate_precision,stage1_gate_recall,stage1_gate_f1,threshold_policy,threshold_recall_floor,threshold,threshold_role,precision,recall,f1,accuracy,tp,fp,fn,tn
0,epoch_summary,1,val,0.710064,0.950151,0.34,0.340000,0.955056,0.501475,0.34,0.340000,0.955056,0.501475,recall_floor,0.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,epoch_summary,2,val,0.720447,0.786514,0.71,0.432836,0.651685,0.520179,0.62,0.378238,0.820225,0.517730,recall_floor,0.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,epoch_summary,3,val,0.816377,0.714872,0.83,0.451327,0.573034,0.504950,0.72,0.384615,0.730337,0.503876,recall_floor,0.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,epoch_summary,4,val,0.646606,0.676037,0.66,0.484615,0.707865,0.575342,0.66,0.484615,0.707865,0.575342,recall_floor,0.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,epoch_summary,5,val,0.637527,0.645798,0.66,0.509259,0.617978,0.558376,0.40,0.406250,0.876404,0.555160,recall_floor,0.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,epoch_summary,6,val,0.640620,0.613468,0.69,0.471429,0.741573,0.576419,0.69,0.471429,0.741573,0.576419,recall_floor,0.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,epoch_summary,7,val,0.720526,0.572264,0.75,0.481481,0.730337,0.580357,0.75,0.481481,0.730337,0.580357,recall_floor,0.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,epoch_summary,8,val,0.697975,0.574593,0.65,0.537037,0.651685,0.588832,0.55,0.440559,0.707865,0.543103,recall_floor,0.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,epoch_summary,9,val,0.721790,0.552421,0.52,0.539130,0.696629,0.607843,0.50,0.508065,0.707865,0.591549,recall_floor,0.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,epoch_summary,10,val,0.709087,0.512929,0.87,0.557692,0.651685,0.601036,0.76,0.454545,0.786517,0.576132,recall_floor,0.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
show_text_file(PROJECT_ROOT / 'runs' / 'two_phase' / 'carry_classifier' / 'summary.md')


# Hold/No-Hold Classifier Training Summary

- Train samples: `3910`
- Validation samples: `496`
- Test samples: `2559`
- Positive weight: `3.4181`
- Best checkpoint epoch: `12`
- Best-F1 validation threshold: `0.83`
- Stage 1 gate threshold: `0.83`
- Threshold policy: `recall_floor`
- Recall floor for gate selection: `0.70`

## Why this threshold was chosen

The real pipeline uses a permissive `hold/no_hold` gate. The selected Stage 1 threshold is the highest validation threshold that still preserves the configured recall floor, with F1 used as a tie-breaker among eligible thresholds.

Expected tradeoff: higher candidate flow to Stage 2, fewer collapsed-recall runs, and more false positives handled later by YOLO26n.

## Validation sweep summary

- Best-F1 validation metrics: precision `0.5508`, recall `0.7303`, F1 `0.6280`
- Gate validation metrics: precision `0.5508`, recall `0.7303`, F1 `0.6280`

| kind            |   epoch | split   |   loss |   train_loss |   best_f1_threshold |   best_f1_precision |   best_f1_recall |   best_f1_f1 |   stage1_gate_threshold |   stage1_gate_precision |   stage1_gate_recall |   stage1_gate_f1 |   threshold_policy |   threshold_recall_floor |   threshold | threshold_role   |   precision |   recall |       f1 |   accuracy |   tp |   fp |   fn |   tn |
|:----------------|--------:|:--------|-------:|-------------:|--------------------:|--------------------:|-----------------:|-------------:|------------------------:|------------------------:|---------------------:|-----------------:|-------------------:|-------------------------:|------------:|:-----------------|------------:|---------:|---------:|-----------:|-----:|-----:|-----:|-----:|
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.05 |                  |    0.286174 | 1        | 0.445    |   0.552419 |   89 |  222 |    0 |  185 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.06 |                  |    0.289902 | 1        | 0.449495 |   0.560484 |   89 |  218 |    0 |  189 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.07 |                  |    0.29085  | 1        | 0.450633 |   0.5625   |   89 |  217 |    0 |  190 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.08 |                  |    0.293729 | 1        | 0.454082 |   0.568548 |   89 |  214 |    0 |  193 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.09 |                  |    0.296667 | 1        | 0.457584 |   0.574597 |   89 |  211 |    0 |  196 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.1  |                  |    0.298658 | 1        | 0.459948 |   0.578629 |   89 |  209 |    0 |  198 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.11 |                  |    0.306897 | 1        | 0.469657 |   0.594758 |   89 |  201 |    0 |  206 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.12 |                  |    0.309028 | 1        | 0.472149 |   0.59879  |   89 |  199 |    0 |  208 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.13 |                  |    0.312281 | 1        | 0.475936 |   0.604839 |   89 |  196 |    0 |  211 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.14 |                  |    0.312057 | 0.988764 | 0.474394 |   0.606855 |   88 |  194 |    1 |  213 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.15 |                  |    0.312057 | 0.988764 | 0.474394 |   0.606855 |   88 |  194 |    1 |  213 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.16 |                  |    0.311828 | 0.977528 | 0.472826 |   0.608871 |   87 |  192 |    2 |  215 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.17 |                  |    0.314079 | 0.977528 | 0.47541  |   0.612903 |   87 |  190 |    2 |  217 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.18 |                  |    0.316364 | 0.977528 | 0.478022 |   0.616935 |   87 |  188 |    2 |  219 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.19 |                  |    0.315018 | 0.966292 | 0.475138 |   0.616935 |   86 |  187 |    3 |  220 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.2  |                  |    0.317343 | 0.966292 | 0.477778 |   0.620968 |   86 |  185 |    3 |  222 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.21 |                  |    0.317343 | 0.966292 | 0.477778 |   0.620968 |   86 |  185 |    3 |  222 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.22 |                  |    0.317343 | 0.966292 | 0.477778 |   0.620968 |   86 |  185 |    3 |  222 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.23 |                  |    0.318519 | 0.966292 | 0.479109 |   0.622984 |   86 |  184 |    3 |  223 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.24 |                  |    0.322097 | 0.966292 | 0.483146 |   0.629032 |   86 |  181 |    3 |  226 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.25 |                  |    0.323308 | 0.966292 | 0.484507 |   0.631048 |   86 |  180 |    3 |  227 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.26 |                  |    0.326996 | 0.966292 | 0.488636 |   0.637097 |   86 |  177 |    3 |  230 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.27 |                  |    0.326923 | 0.955056 | 0.487106 |   0.639113 |   85 |  175 |    4 |  232 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.28 |                  |    0.330739 | 0.955056 | 0.491329 |   0.645161 |   85 |  172 |    4 |  235 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.29 |                  |    0.328125 | 0.94382  | 0.486957 |   0.643145 |   84 |  172 |    5 |  235 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.3  |                  |    0.326772 | 0.932584 | 0.483965 |   0.643145 |   83 |  171 |    6 |  236 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.31 |                  |    0.333333 | 0.932584 | 0.491124 |   0.653226 |   83 |  166 |    6 |  241 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.32 |                  |    0.337398 | 0.932584 | 0.495522 |   0.659274 |   83 |  163 |    6 |  244 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.33 |                  |    0.340164 | 0.932584 | 0.498498 |   0.663306 |   83 |  161 |    6 |  246 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.34 |                  |    0.342975 | 0.932584 | 0.501511 |   0.667339 |   83 |  159 |    6 |  248 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.35 |                  |    0.350211 | 0.932584 | 0.509202 |   0.677419 |   83 |  154 |    6 |  253 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.36 |                  |    0.350211 | 0.932584 | 0.509202 |   0.677419 |   83 |  154 |    6 |  253 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.37 |                  |    0.350211 | 0.932584 | 0.509202 |   0.677419 |   83 |  154 |    6 |  253 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.38 |                  |    0.347458 | 0.921348 | 0.504615 |   0.675403 |   82 |  154 |    7 |  253 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.39 |                  |    0.348936 | 0.921348 | 0.506173 |   0.677419 |   82 |  153 |    7 |  254 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.4  |                  |    0.350427 | 0.921348 | 0.50774  |   0.679435 |   82 |  152 |    7 |  255 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.41 |                  |    0.351931 | 0.921348 | 0.509317 |   0.681452 |   82 |  151 |    7 |  256 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.42 |                  |    0.350649 | 0.910112 | 0.50625  |   0.681452 |   81 |  150 |    8 |  257 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.43 |                  |    0.352174 | 0.910112 | 0.507837 |   0.683468 |   81 |  149 |    8 |  258 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.44 |                  |    0.356828 | 0.910112 | 0.512658 |   0.689516 |   81 |  146 |    8 |  261 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.45 |                  |    0.356828 | 0.910112 | 0.512658 |   0.689516 |   81 |  146 |    8 |  261 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.46 |                  |    0.361607 | 0.910112 | 0.517572 |   0.695565 |   81 |  143 |    8 |  264 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.47 |                  |    0.361607 | 0.910112 | 0.517572 |   0.695565 |   81 |  143 |    8 |  264 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.48 |                  |    0.364865 | 0.910112 | 0.5209   |   0.699597 |   81 |  141 |    8 |  266 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.49 |                  |    0.366516 | 0.910112 | 0.522581 |   0.701613 |   81 |  140 |    8 |  267 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.5  |                  |    0.373272 | 0.910112 | 0.529412 |   0.709677 |   81 |  136 |    8 |  271 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.51 |                  |    0.375    | 0.910112 | 0.531148 |   0.711694 |   81 |  135 |    8 |  272 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.52 |                  |    0.376744 | 0.910112 | 0.532895 |   0.71371  |   81 |  134 |    8 |  273 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.53 |                  |    0.377358 | 0.898876 | 0.531561 |   0.715726 |   80 |  132 |    9 |  275 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.54 |                  |    0.377358 | 0.898876 | 0.531561 |   0.715726 |   80 |  132 |    9 |  275 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.55 |                  |    0.37619  | 0.88764  | 0.528428 |   0.715726 |   79 |  131 |   10 |  276 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.56 |                  |    0.379808 | 0.88764  | 0.531987 |   0.719758 |   79 |  129 |   10 |  278 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.57 |                  |    0.383495 | 0.88764  | 0.535593 |   0.72379  |   79 |  127 |   10 |  280 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.58 |                  |    0.391089 | 0.88764  | 0.542955 |   0.731855 |   79 |  123 |   10 |  284 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.59 |                  |    0.395    | 0.88764  | 0.546713 |   0.735887 |   79 |  121 |   10 |  286 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.6  |                  |    0.401015 | 0.88764  | 0.552448 |   0.741935 |   79 |  118 |   10 |  289 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.61 |                  |    0.407216 | 0.88764  | 0.558304 |   0.747984 |   79 |  115 |   10 |  292 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.62 |                  |    0.40625  | 0.876404 | 0.55516  |   0.747984 |   78 |  114 |   11 |  293 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.63 |                  |    0.410526 | 0.876404 | 0.55914  |   0.752016 |   78 |  112 |   11 |  295 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.64 |                  |    0.411765 | 0.865169 | 0.557971 |   0.754032 |   77 |  110 |   12 |  297 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.65 |                  |    0.418478 | 0.865169 | 0.564103 |   0.760081 |   77 |  107 |   12 |  300 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.66 |                  |    0.420765 | 0.865169 | 0.566176 |   0.762097 |   77 |  106 |   12 |  301 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.67 |                  |    0.425414 | 0.865169 | 0.57037  |   0.766129 |   77 |  104 |   12 |  303 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.68 |                  |    0.429379 | 0.853933 | 0.571429 |   0.770161 |   76 |  101 |   13 |  306 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.69 |                  |    0.439306 | 0.853933 | 0.580153 |   0.778226 |   76 |   97 |   13 |  310 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.7  |                  |    0.449102 | 0.842697 | 0.585938 |   0.78629  |   75 |   92 |   14 |  315 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.71 |                  |    0.454545 | 0.842697 | 0.590551 |   0.790323 |   75 |   90 |   14 |  317 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.72 |                  |    0.460123 | 0.842697 | 0.595238 |   0.794355 |   75 |   88 |   14 |  319 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.73 |                  |    0.458599 | 0.808989 | 0.585366 |   0.794355 |   72 |   85 |   17 |  322 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.74 |                  |    0.461538 | 0.808989 | 0.587755 |   0.796371 |   72 |   84 |   17 |  323 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.75 |                  |    0.467532 | 0.808989 | 0.592593 |   0.800403 |   72 |   82 |   17 |  325 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.76 |                  |    0.473684 | 0.808989 | 0.59751  |   0.804435 |   72 |   80 |   17 |  327 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.77 |                  |    0.472973 | 0.786517 | 0.590717 |   0.804435 |   70 |   78 |   19 |  329 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.78 |                  |    0.492958 | 0.786517 | 0.606061 |   0.816532 |   70 |   72 |   19 |  335 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.79 |                  |    0.496403 | 0.775281 | 0.605263 |   0.818548 |   69 |   70 |   20 |  337 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.8  |                  |    0.515152 | 0.764045 | 0.615385 |   0.828629 |   68 |   64 |   21 |  343 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.81 |                  |    0.532258 | 0.741573 | 0.619718 |   0.836694 |   66 |   58 |   23 |  349 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.82 |                  |    0.528455 | 0.730337 | 0.613208 |   0.834677 |   65 |   58 |   24 |  349 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.83 | best_f1_and_gate |    0.550847 | 0.730337 | 0.628019 |   0.844758 |   65 |   53 |   24 |  354 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.84 |                  |    0.547009 | 0.719101 | 0.621359 |   0.842742 |   64 |   53 |   25 |  354 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.85 |                  |    0.558559 | 0.696629 | 0.62     |   0.846774 |   62 |   49 |   27 |  358 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.86 |                  |    0.550459 | 0.674157 | 0.606061 |   0.842742 |   60 |   49 |   29 |  358 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.87 |                  |    0.542056 | 0.651685 | 0.591837 |   0.83871  |   58 |   49 |   31 |  358 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.88 |                  |    0.558824 | 0.640449 | 0.596859 |   0.844758 |   57 |   45 |   32 |  362 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.89 |                  |    0.56     | 0.629213 | 0.592593 |   0.844758 |   56 |   44 |   33 |  363 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.9  |                  |    0.5625   | 0.606742 | 0.583784 |   0.844758 |   54 |   42 |   35 |  365 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.91 |                  |    0.597701 | 0.58427  | 0.590909 |   0.854839 |   52 |   35 |   37 |  372 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.92 |                  |    0.628205 | 0.550562 | 0.586826 |   0.860887 |   49 |   29 |   40 |  378 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.93 |                  |    0.657143 | 0.516854 | 0.578616 |   0.864919 |   46 |   24 |   43 |  383 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.94 |                  |    0.693548 | 0.483146 | 0.569536 |   0.868952 |   43 |   19 |   46 |  388 |
| threshold_sweep |      12 | val     |    nan |          nan |                 nan |                 nan |              nan |          nan |                     nan |                     nan |                  nan |              nan |                nan |                      nan |        0.95 |                  |    0.701754 | 0.449438 | 0.547945 |   0.866935 |   40 |   17 |   49 |  390 |

## Final test metrics with gate threshold

| kind       |   epoch | split   |    loss | train_loss   |   best_f1_threshold |   best_f1_precision |   best_f1_recall |   best_f1_f1 |   stage1_gate_threshold |   stage1_gate_precision |   stage1_gate_recall |   stage1_gate_f1 | threshold_policy   |   threshold_recall_floor |   threshold |   threshold_role |   precision |   recall |       f1 |   accuracy |   tp |   fp |   fn |   tn |
|:-----------|--------:|:--------|--------:|:-------------|--------------------:|--------------------:|-----------------:|-------------:|------------------------:|------------------------:|---------------------:|-----------------:|:-------------------|-------------------------:|------------:|-----------------:|------------:|---------:|---------:|-----------:|-----:|-----:|-----:|-----:|
| final_test |      12 | test    | 2.03513 |              |                0.83 |                 nan |              nan |          nan |                    0.83 |                     nan |                  nan |              nan | recall_floor       |                      0.7 |         nan |              nan |    0.747945 | 0.372442 | 0.497268 |   0.568581 |  546 |  184 |  920 |  909 |


## Step 3. Run two-phase inference

This uses:

- the person detector from `configs/two_phase.yaml` unless you override it
- your crop-stage YOLO `best.pt` checkpoint as the Stage 2 weapon detector
- the trained `hold` / `no_hold` classifier checkpoint only when `ENABLE_HOLD_GATE = True`


In [16]:
cmd = [
    sys.executable,
    'scripts/run_two_phase_inference.py',
    '--split', SPLIT,
    '--device', DEVICE,
    '--weapon-model', str(WEAPON_MODEL),
    '--output-prefix', OUTPUT_PREFIX,
]

if PERSON_MODEL is not None:
    cmd.extend(['--person-model', PERSON_MODEL])

if ENABLE_HOLD_GATE:
    cmd.append('--enable-hold-gate')
    if Path(HOLD_CHECKPOINT).exists():
        cmd.extend(['--hold-checkpoint', str(HOLD_CHECKPOINT)])

run_command(cmd)


Running:
.venv\Scripts\python.exe scripts/run_two_phase_inference.py --split test --device 0 --weapon-model runs/single_stage\yolo26n_img9604\weights\best.pt --output-prefix test --hold-checkpoint runs/two_phase\carry_classifier\best.pt

STDOUT:
[OK] Saved predictions: runs/two_phase\predictions\test_predictions.csv
[OK] Saved image summary: runs/two_phase\predictions\test_image_summary.csv
[OK] Saved pipeline summary: runs/two_phase\predictions\test_pipeline_summary.csv
[OK] Hold threshold used: 0.83 (checkpoint_stage1_gate_threshold)

STDERR:

RETURN CODE: 0


In [17]:
print('Predictions preview')
display(show_csv(TWO_PHASE_PREDICTIONS, rows=10))

print('\nImage summary preview')
display(show_csv(TWO_PHASE_IMAGE_SUMMARY, rows=10))

print('\nPipeline summary preview')
display(show_csv(TWO_PHASE_PIPELINE_SUMMARY, rows=10))


Predictions preview
runs/two_phase\predictions\test_predictions.csv -> 461 rows


,split,image_stem,image_path,person_index,weapon_index,person_confidence,hold_probability,carry_probability,weapon_confidence,xmin,ymin,xmax,ymax
0,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data/raw\images\Cam5-From09-24-18To09-26-43-Gu...,3,0,0.848314,0.900184,0.900184,0.562785,633.027,404.497,651.954,439.705
1,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data/raw\images\Cam5-From09-24-18To09-26-43-Gu...,3,1,0.848314,0.900184,0.900184,0.541825,495.409,384.865,512.403,430.856
2,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data/raw\images\Cam5-From09-24-18To09-26-43-Gu...,3,0,0.817063,0.977770,0.977770,0.760882,520.303,371.892,535.147,413.323
3,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data/raw\images\Cam5-From09-24-18To09-26-43-Gu...,3,1,0.817063,0.977770,0.977770,0.513782,628.887,408.225,646.342,445.778
4,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data/raw\images\Cam5-From09-24-18To09-26-43-Gu...,3,0,0.827574,0.980258,0.980258,0.669767,513.789,368.281,526.883,408.137
5,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data/raw\images\Cam5-From09-24-18To09-26-43-Gu...,6,0,0.513124,0.847190,0.847190,0.307904,25.570,415.601,50.550,428.933
6,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data/raw\images\Cam5-From09-24-18To09-26-43-Gu...,3,1,0.827574,0.980258,0.980258,0.252982,624.301,410.344,640.769,447.170
7,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data/raw\images\Cam5-From09-24-18To09-26-43-Gu...,3,0,0.833507,0.955572,0.955572,0.433101,622.739,406.361,642.118,448.126
8,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data/raw\images\Cam5-From09-24-18To09-26-43-Gu...,4,0,0.804236,0.989585,0.989585,0.786065,515.618,377.227,530.655,416.823
9,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data/raw\images\Cam5-From09-24-18To09-26-43-Gu...,4,1,0.804236,0.989585,0.989585,0.755173,631.074,405.297,651.539,442.819



Image summary preview
runs/two_phase\predictions\test_image_summary.csv -> 1031 rows


,split,image_stem,image_path,stage0_image_miss,persons_detected,persons_rejected_by_hold_gate,persons_filtered_out,persons_passed_stage2,final_weapon_detections
0,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data/raw\images\Cam5-From09-24-18To09-26-43-Gu...,0,0,0,0,0,0
1,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data/raw\images\Cam5-From09-24-18To09-26-43-Gu...,0,2,1,1,1,0
2,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data/raw\images\Cam5-From09-24-18To09-26-43-Gu...,0,7,6,6,1,2
3,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data/raw\images\Cam5-From09-24-18To09-26-43-Gu...,0,6,5,5,1,2
4,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data/raw\images\Cam5-From09-24-18To09-26-43-Gu...,0,7,5,5,2,3
5,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data/raw\images\Cam5-From09-24-18To09-26-43-Gu...,0,6,6,6,0,0
6,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data/raw\images\Cam5-From09-24-18To09-26-43-Gu...,0,6,5,5,1,1
7,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data/raw\images\Cam5-From09-24-18To09-26-43-Gu...,0,8,7,7,1,3
8,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data/raw\images\Cam5-From09-24-18To09-26-43-Gu...,0,9,8,8,1,2
9,test,Cam5-From09-24-18To09-26-43-Guns_x264_Segment_...,data/raw\images\Cam5-From09-24-18To09-26-43-Gu...,0,8,6,6,2,2



Pipeline summary preview
runs/two_phase\predictions\test_pipeline_summary.csv -> 1 rows


,split,images_processed,persons_detected,persons_rejected_by_hold_gate,persons_filtered_out,persons_passed_stage2,final_weapon_detections,stage0_miss_images,hold_threshold,carry_threshold,threshold_source
0,test,1031,2785,2001,2001,784,461,42,0.83,0.83,checkpoint_stage1_gate_threshold


## Step 4. Compare the two-phase pipeline against the single-stage YOLO26n baseline

The same YOLO26n checkpoint is used as the single-stage baseline model on the full image.


In [18]:
cmd = [
    sys.executable,
    'scripts/evaluate_detection_pipeline.py',
    '--split', SPLIT,
    '--device', DEVICE,
    '--single-stage-model', str(WEAPON_MODEL),
    '--two-phase-predictions', str(TWO_PHASE_PREDICTIONS),
    '--two-phase-image-summary', str(TWO_PHASE_IMAGE_SUMMARY),
    '--output-prefix', OUTPUT_PREFIX,
]

run_command(cmd)


Running:
.venv\Scripts\python.exe scripts/evaluate_detection_pipeline.py --split test --device 0 --single-stage-model runs/single_stage\yolo26n_img9604\weights\best.pt --two-phase-predictions runs/two_phase\predictions\test_predictions.csv --two-phase-image-summary runs/two_phase\predictions\test_image_summary.csv --output-prefix test

STDOUT:
[OK] Saved comparison CSV: runs/two_phase\evaluation\test_comparison.csv
[OK] Saved summary: runs/two_phase\evaluation\test_summary.md

STDERR:

RETURN CODE: 0


In [19]:
comparison_df = pd.read_csv(EVAL_COMPARISON)
display(comparison_df)


,pipeline,tp,fp,fn,precision,recall,f1,detections_per_image,stage0_miss_count,stage1_rejected,stage2_passed
0,single_stage,232,78,1281,0.748387,0.153338,0.254526,0.300679,NaN,NaN,NaN
1,two_phase,54,407,1459,0.117137,0.035691,0.054711,0.447139,42.0,2001.0,784.0
2,delta_two_phase_minus_single_stage,-178,329,178,-0.631250,-0.117647,-0.199814,0.146460,42.0,2001.0,784.0


In [20]:
show_text_file(EVAL_SUMMARY)


# Detection Pipeline Evaluation Summary

- Split: `test`
- Evaluation IoU threshold: `0.50`

## Comparison

| pipeline                           |   tp |   fp |   fn |   precision |     recall |         f1 |   detections_per_image |   stage0_miss_count |   stage1_rejected |   stage2_passed |
|:-----------------------------------|-----:|-----:|-----:|------------:|-----------:|-----------:|-----------------------:|--------------------:|------------------:|----------------:|
| single_stage                       |  232 |   78 | 1281 |    0.748387 |  0.153338  |  0.254526  |               0.300679 |                     |                   |                 |
| two_phase                          |   54 |  407 | 1459 |    0.117137 |  0.0356907 |  0.0547112 |               0.447139 |                  42 |              2001 |             784 |
| delta_two_phase_minus_single_stage | -178 |  329 |  178 |   -0.63125  | -0.117647  | -0.199814  |               0.14646  |                  42 |              2001 |             784 |


## Output files

After a successful run, the main artifacts are:

- `data/interim/two_phase/crops/`
- `data/interim/two_phase/metadata/`
- `runs/two_phase/carry_classifier/best.pt`
- `runs/two_phase/predictions/{split}_predictions.csv`
- `runs/two_phase/predictions/{split}_image_summary.csv`
- `runs/two_phase/evaluation/{split}_comparison.csv`
